In [0]:
from pyspark.sql import functions as F

In [0]:
%run /Workspace/Users/yanquiel@softserve.academy/ecommerce-bronze-platform-/notebooks/utilities

In [0]:
dbutils.widgets.text("catalog", "dbr_dev", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")


catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")



In [0]:
volume_name = "landing"

source_path = f"/Volumes/{catalog}/{bronze_schema}/{volume_name}/{data_source}/*.csv"

In [0]:
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(source_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
(df.write
 .format("delta")
 .option("delta.enableChangeDataFeed", "true")
 .mode("overwrite")
 .saveAsTable(f"{catalog}.{bronze_schema}.brz_{data_source}")
)